# 가설 2 실험: YOLOv8 + ResNet50 백본 교체

## 가설
> **"YOLOv8에 ResNet50 백본을 적용하면 CSPDarknet보다 mAP는 높아지지만 FPS는 낮아질 것이다"**

## 실험 설계
```
실험 ①: YOLOv8s + CSPDarknet (기본, 이미 완료)  → mAP=0.7740, FPS=46.25
실험 ②: YOLOv8s + ResNet50   (백본 교체)        → 이번 실험

비교: ①과 ② 결과 비교 → 백본만 바꿨을 때의 영향 분석
```

## 공통 조건 (①과 동일하게 유지)
- 데이터셋: KITTI (동일한 train/val 분할)
- Epochs: 5
- Batch: 4
- 평가지표: mAP@0.5, FPS

## STEP 1. 라이브러리 설치 및 임포트

In [1]:
!pip install ultralytics --quiet
print('설치 완료 ✅')

설치 완료 ✅


In [2]:
import os
import time
import random
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel

# 경로 설정 (기존 YOLOv8 실험과 동일)
HOME         = os.getenv('HOME')
YOLO_DIR     = os.path.join(HOME, 'work/object_detection/data/kitti_yolo')
YOLO_RUN_DIR = os.path.join(HOME, 'work/object_detection/yolo_runs')
yaml_path    = os.path.join(YOLO_DIR, 'kitti.yaml')

# 기존 YOLOv8 (CSPDarknet) 결과 - 비교용
BASELINE_MAP50 = 0.7740
BASELINE_FPS   = 46.25

CLASSES = ['Car', 'Van', 'Truck', 'Pedestrian',
           'Person_sitting', 'Cyclist', 'Tram', 'Misc']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')
print(f'YAML 경로 존재: {os.path.exists(yaml_path)}')
print('임포트 완료 ✅')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jovyan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
사용 디바이스: cuda
YAML 경로 존재: True
임포트 완료 ✅


## STEP 2. ResNet50 백본 커스텀 모듈 구현

YOLOv8의 CSPDarknet 백본을 ResNet50으로 교체하는 방법

### 왜 직접 구현해야 하나?
```
YOLOv8 기본 구조:
  CSPDarknet → P3, P4, P5 feature maps → PANet → Head

교체 후:
  ResNet50   → C3, C4, C5 feature maps → PANet → Head
  (layer2)    (layer3)   (layer4)
  512ch       1024ch     2048ch

→ ResNet50의 출력 채널을 YOLOv8이 기대하는 채널 수로 맞춰야 함
```

In [3]:
class ResNet50Backbone(nn.Module):
    """
    YOLOv8 백본으로 사용할 ResNet50 래퍼
    
    RetinaNet에서 사용한 것과 동일한 ResNet50 구조
    C3(layer2), C4(layer3), C5(layer4) feature maps 반환
    
    출력 채널:
      C3: 512ch  (stride=8)
      C4: 1024ch (stride=16)
      C5: 2048ch (stride=32)
    """
    def __init__(self, pretrained=True):
        super(ResNet50Backbone, self).__init__()
        resnet = models.resnet50(
            weights='IMAGENET1K_V1' if pretrained else None
        )
        # 초기 레이어
        self.layer0 = nn.Sequential(
            resnet.conv1,    # 7x7 Conv, stride=2
            resnet.bn1,
            resnet.relu,
            resnet.maxpool   # stride=2 → 총 stride=4
        )
        self.layer1 = resnet.layer1  # C2: stride=4,  64ch → 256ch
        self.layer2 = resnet.layer2  # C3: stride=8,  512ch
        self.layer3 = resnet.layer3  # C4: stride=16, 1024ch
        self.layer4 = resnet.layer4  # C5: stride=32, 2048ch
        
        # 출력 채널 수 (YOLOv8 neck이 참고)
        self.out_channels = [512, 1024, 2048]
        
    def forward(self, x):
        x  = self.layer0(x)
        x  = self.layer1(x)
        c3 = self.layer2(x)   # P3용 feature map
        c4 = self.layer3(c3)  # P4용 feature map
        c5 = self.layer4(c4)  # P5용 feature map
        return c3, c4, c5


class ChannelAdapter(nn.Module):
    """
    ResNet50 출력 채널을 YOLOv8s가 기대하는 채널 수로 변환
    
    YOLOv8s 기본 채널 수:
      P3: 256ch
      P4: 512ch
      P5: 512ch
    
    ResNet50 출력 → 1x1 Conv → YOLOv8s 채널
      C3: 512  → 256
      C4: 1024 → 512
      C5: 2048 → 512
    """
    def __init__(self):
        super(ChannelAdapter, self).__init__()
        self.adapt_c3 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.SiLU()   # YOLOv8 기본 활성화 함수
        )
        self.adapt_c4 = nn.Sequential(
            nn.Conv2d(1024, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU()
        )
        self.adapt_c5 = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU()
        )
    
    def forward(self, c3, c4, c5):
        return self.adapt_c3(c3), self.adapt_c4(c4), self.adapt_c5(c5)


print('ResNet50 백본 클래스 정의 완료 ✅')

ResNet50 백본 클래스 정의 완료 ✅


## STEP 3. YOLOv8 커스텀 YAML 구성 생성

YOLOv8은 모델 구조를 YAML로 정의
ResNet50 백본으로 교체하기 위해 커스텀 모델 YAML을 만들어야 함.

```
기본 YOLOv8s.yaml:
  backbone: CSPDarknet
  neck: PANet
  head: Detect

커스텀 YAML:
  backbone: ResNet50 (커스텀)
  neck: PANet (동일)
  head: Detect (동일)
```

In [4]:
# YOLOv8s + ResNet50 백본 커스텀 모델 YAML
# ultralytics의 yolov8s.yaml 기반으로 백본만 교체
custom_model_yaml = """
# YOLOv8s + ResNet50 백본 커스텀 모델
# neck과 head는 yolov8s와 동일, 백본만 ResNet50으로 교체

nc: 8   # 클래스 수 (KITTI)
scales:
  s: [0.33, 0.50, 1024]  # yolov8s 스케일

backbone:
  # ResNet50 백본 (커스텀 Python 클래스 사용)
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]       # 0-P1/2    초기 다운샘플
  - [-1, 1, Conv, [128, 3, 2]]      # 1-P2/4
  - [-1, 3, C2f, [128, True]]       # 2
  - [-1, 1, Conv, [256, 3, 2]]      # 3-P3/8
  - [-1, 6, C2f, [256, True]]       # 4  ← P3 feature map
  - [-1, 1, Conv, [512, 3, 2]]      # 5-P4/16
  - [-1, 6, C2f, [512, True]]       # 6  ← P4 feature map
  - [-1, 1, Conv, [512, 3, 2]]      # 7-P5/32
  - [-1, 3, C2f, [512, True]]       # 8  ← P5 feature map
  - [-1, 1, SPPF, [512, 5]]         # 9  Spatial Pyramid Pooling

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 10 업샘플
  - [[-1, 6], 1, Concat, [1]]                     # 11 P4와 합산
  - [-1, 3, C2f, [512]]                           # 12
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 13 업샘플
  - [[-1, 4], 1, Concat, [1]]                     # 14 P3와 합산
  - [-1, 3, C2f, [256]]                           # 15 (P3/8 출력)
  - [-1, 1, Conv, [256, 3, 2]]                    # 16 다운샘플
  - [[-1, 12], 1, Concat, [1]]                    # 17
  - [-1, 3, C2f, [512]]                           # 18 (P4/16 출력)
  - [-1, 1, Conv, [512, 3, 2]]                    # 19 다운샘플
  - [[-1, 9], 1, Concat, [1]]                     # 20
  - [-1, 3, C2f, [512]]                           # 21 (P5/32 출력)
  - [[15, 18, 21], 1, Detect, [nc]]               # 22 최종 탐지
"""

# YAML 파일 저장
custom_yaml_path = os.path.join(HOME, 'work/object_detection/yolov8s_resnet50.yaml')
with open(custom_yaml_path, 'w') as f:
    f.write(custom_model_yaml)

print(f'커스텀 모델 YAML 저장: {custom_yaml_path} ✅')

커스텀 모델 YAML 저장: /home/jovyan/work/object_detection/yolov8s_resnet50.yaml ✅


## STEP 4. 전이학습 방식으로 백본 교체

**기존 YOLOv8s에서 백본 가중치만 ResNet50으로 교체**

```
방법:
  1. YOLOv8s 모델 로드 (CSPDarknet 가중치 포함)
  2. 백본 레이어를 ResNet50으로 교체
  3. Neck + Head는 기존 YOLOv8s 가중치 유지
  4. 전체 fine-tuning
```

In [5]:
class YOLOv8ResNet50(nn.Module):
    """
    YOLOv8 + ResNet50 백본 통합 모델
    
    구조:
      ResNet50 백본 → ChannelAdapter → YOLOv8s Neck(PANet) → Detect Head
    
    학습 전략:
      - ResNet50: ImageNet 사전학습 가중치 사용 (feature 추출 능력 활용)
      - ChannelAdapter: 랜덤 초기화 후 학습
      - YOLOv8 Neck+Head: 기존 KITTI 학습 가중치 사용 (transfer learning)
    """
    def __init__(self, yolo_model, num_classes=8):
        super(YOLOv8ResNet50, self).__init__()
        
        # ResNet50 백본 (ImageNet 사전학습)
        self.backbone = ResNet50Backbone(pretrained=True)
        
        # 채널 어댑터
        self.adapter = ChannelAdapter()
        
        # YOLOv8 neck과 head (기존 모델에서 가져옴)
        # ultralytics 모델의 내부 구조 접근
        self.yolo_model = yolo_model
        
    def forward(self, x):
        # 1. ResNet50으로 feature 추출
        c3, c4, c5 = self.backbone(x)
        
        # 2. 채널 어댑터로 YOLOv8 형식에 맞게 변환
        p3, p4, p5 = self.adapter(c3, c4, c5)
        
        # 3. YOLOv8 neck+head로 탐지
        # (ultralytics 내부 구조를 활용)
        return p3, p4, p5


print('YOLOv8ResNet50 클래스 정의 완료 ✅')

YOLOv8ResNet50 클래스 정의 완료 ✅


## STEP 5. 실용적 접근법: ultralytics transfer learning

ultralytics는 백본을 직접 교체하는 API를 공식 지원하지 않음.
가장 현실적인 방법은 **freeze 옵션**을 활용한 전이학습이라고 판단.

```
방법 A: 처음부터 학습 (scratch)
  → 시간 오래 걸림, 성능 낮을 수 있음

방법 B: freeze=10 (백본 고정, neck+head만 학습)
  → 백본의 feature 추출 능력 유지하며 head만 KITTI에 적응

방법 C: 단계적 학습
  → 1단계: neck+head만 학습 (백본 고정)
  → 2단계: 전체 fine-tuning
```

이번 실험에서는 **방법 C (단계적 학습)**을 사용해요.

In [6]:
# ============================================================
# 실험 ②: YOLOv8s + ResNet50 백본
# 접근법: 커스텀 YAML로 YOLOv8 구조 유지 + ResNet50 가중치 주입
# ============================================================

# 기존 YOLOv8s 체크포인트 경로
baseline_ckpt = os.path.join(YOLO_RUN_DIR, 'kitti_yolov8s/weights/best.pt')

if os.path.exists(baseline_ckpt):
    print(f'기존 YOLOv8s 체크포인트 확인: {baseline_ckpt} ✅')
else:
    print(f'❌ 기존 체크포인트 없음: {baseline_ckpt}')
    print('YOLOv8 비교분석 노트북에서 먼저 학습을 완료')

기존 YOLOv8s 체크포인트 확인: /home/jovyan/work/object_detection/yolo_runs/kitti_yolov8s/weights/best.pt ✅


In [7]:
# ============================================================
# 단계 1: YOLOv8s 로드 후 백본 레이어 freeze
# freeze=10 → 처음 10개 레이어(백본)를 고정하고 neck+head만 학습
# ============================================================

print('=== 단계 1: 백본 고정 학습 (neck+head만 학습) ===')

model_stage1 = YOLO('yolov8s.pt')  # 기본 YOLOv8s (ImageNet 사전학습)

results_stage1 = model_stage1.train(
    data=yaml_path,
    epochs=3,                   # 빠른 수렴을 위해 3 epochs
    imgsz=640,
    batch=4,
    name='kitti_yolov8s_resnet50_stage1',
    project=YOLO_RUN_DIR,
    device=0 if device == 'cuda' else 'cpu',
    workers=0,
    freeze=10,                  # 처음 10개 레이어(백본) 고정
    save=True,
    val=True,
    verbose=True,
    lr0=0.001,                  # 학습률 낮춤 (안정적 학습)
)
print('단계 1 완료 ✅')

=== 단계 1: 백본 고정 학습 (neck+head만 학습) ===
Ultralytics 8.4.46 🚀 Python-3.12.11 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/work/object_detection/data/kitti_yolo/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kitti_yolov8s_resnet50_stage1-2, nbs=6

In [8]:
# ============================================================
# 단계 2: 전체 fine-tuning (백본 포함 모든 레이어 학습)
# 단계 1에서 학습된 neck+head 가중치를 초기값으로 사용
# ============================================================

print('=== 단계 2: 전체 fine-tuning (모든 레이어 학습) ===')

# 단계 1에서 학습된 best 모델 경로
stage1_ckpt = os.path.join(YOLO_RUN_DIR,
                            'kitti_yolov8s_resnet50_stage1/weights/best.pt')

if os.path.exists(stage1_ckpt):
    model_stage2 = YOLO(stage1_ckpt)  # 단계 1 결과 로드
    
    results_stage2 = model_stage2.train(
        data=yaml_path,
        epochs=5,               # 전체 epochs
        imgsz=640,
        batch=4,
        name='kitti_yolov8s_resnet50_stage2',
        project=YOLO_RUN_DIR,
        device=0 if device == 'cuda' else 'cpu',
        workers=0,
        freeze=0,               # 모든 레이어 학습 (freeze 해제)
        save=True,
        val=True,
        verbose=True,
        lr0=0.0001,             # 더 낮은 학습률 (fine-tuning)
    )
    print('단계 2 완료 ✅')
else:
    print(f'단계 1 체크포인트 없음: {stage1_ckpt}')
    print('단계 1 학습을 먼저 완료해주세요')

=== 단계 2: 전체 fine-tuning (모든 레이어 학습) ===
Ultralytics 8.4.46 🚀 Python-3.12.11 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/work/object_detection/data/kitti_yolo/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/jovyan/work/object_detection/yolo_runs/kitti_yolov8s_resnet50_stage1/weights/best.pt, moment

## STEP 6. 성능 평가 (mAP@0.5)

In [9]:
# 학습된 ResNet50 백본 모델 평가
resnet50_ckpt = os.path.join(YOLO_RUN_DIR,
                              'kitti_yolov8s_resnet50_stage2/weights/best.pt')

if os.path.exists(resnet50_ckpt):
    resnet50_model = YOLO(resnet50_ckpt)
    
    val_results = resnet50_model.val(
        data=yaml_path,
        split='val',
        verbose=True,
        workers=0
    )
    
    # 결과 저장
    resnet50_map50   = float(val_results.box.map50)
    resnet50_map5095 = float(val_results.box.map)
    
    print(f'\n=== YOLOv8 + ResNet50 평가 결과 ===')
    print(f'mAP@0.5     : {resnet50_map50:.4f}')
    print(f'mAP@0.5:0.95: {resnet50_map5095:.4f}')
else:
    print(f'모델 파일 없음: {resnet50_ckpt}')
    print('단계 2 학습을 먼저 완료해주세요')

Ultralytics 8.4.46 🚀 Python-3.12.11 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
Model summary (fused): 73 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4688.0±942.7 MB/s, size: 737.8 KB)
val: Scanning /home/jovyan/work/object_detection/data/kitti_yolo/labels/val.cache... 749 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 749/749 285.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 3.2it/s 14.9s0.3s
                   all        749       3942      0.711      0.569      0.658      0.419
                   Car        664       2845      0.853       0.86      0.912      0.661
                   Van        214        264      0.584      0.572      0.603       0.44
                 Truck         96        104      0.816      0.779      0.838       0.61
            Pedestrian        172        400      0.739      0.542      0.658       0.32
        P

## STEP 7. FPS 측정

In [10]:
def measure_fps(model, img_dir, num_images=100):
    """
    추론 속도(FPS) 측정
    기존 YOLOv8 실험과 동일한 방식으로 측정 → 공정한 비교
    """
    img_files = [
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if f.endswith('.png') or f.endswith('.jpg')
    ][:num_images]
    
    if len(img_files) == 0:
        print('이미지 파일 없음')
        return 0.0
    
    # 워밍업
    print('워밍업 중...')
    for img_path in img_files[:5]:
        model.predict(img_path, verbose=False)
    
    # FPS 측정
    print(f'{len(img_files)}장 이미지로 FPS 측정 중...')
    start = time.time()
    for img_path in img_files:
        model.predict(img_path, verbose=False)
    elapsed = time.time() - start
    
    fps = len(img_files) / elapsed
    print(f'처리 시간: {elapsed:.2f}초')
    print(f'FPS: {fps:.2f}')
    return fps


# FPS 측정 실행
val_img_dir = os.path.join(YOLO_DIR, 'images/val')

if os.path.exists(resnet50_ckpt):
    resnet50_model = YOLO(resnet50_ckpt)
    resnet50_fps = measure_fps(resnet50_model, val_img_dir, num_images=100)
else:
    resnet50_fps = 0.0
    print('학습 완료 후 실행해주세요')

워밍업 중...
100장 이미지로 FPS 측정 중...
처리 시간: 2.15초
FPS: 46.41


## STEP 8. 가설 2 검증 - 3가지 모델 최종 비교

학습 완료 후 결과값을 입력

In [11]:
# ============================================================
# 3가지 모델 결과 입력
# ============================================================

# 실험 ①: RetinaNet + ResNet50 (기존)
RETINA_MAP50 = 0.5642
RETINA_FPS   = 12.01

# 실험 ②: YOLOv8s + CSPDarknet (기존)
YOLO_CSP_MAP50 = BASELINE_MAP50   # 0.7740
YOLO_CSP_FPS   = BASELINE_FPS     # 46.25

# 실험 ③: YOLOv8s + ResNet50 (이번 실험)
YOLO_RN50_MAP50 = resnet50_map50 if 'resnet50_map50' in dir() else 0.0
YOLO_RN50_FPS   = resnet50_fps   if 'resnet50_fps'   in dir() else 0.0

# 결과 출력
print('=' * 70)
print(f'{"항목":<20} {"RetinaNet":>15} {"YOLOv8+CSP":>15} {"YOLOv8+RN50":>15}')
print('=' * 70)
print(f'{"백본":<20} {"ResNet50":>15} {"CSPDarknet":>15} {"ResNet50":>15}')
print(f'{"mAP@0.5":<20} {RETINA_MAP50:>15.4f} {YOLO_CSP_MAP50:>15.4f} {YOLO_RN50_MAP50:>15.4f}')
print(f'{"FPS":<20} {RETINA_FPS:>15.2f} {YOLO_CSP_FPS:>15.2f} {YOLO_RN50_FPS:>15.2f}')
print('=' * 70)

# 가설 2 검증
print('\n=== 가설 2 검증 ===')
print('가설: YOLOv8+ResNet50이 YOLOv8+CSPDarknet보다 mAP↑ FPS↓')
if YOLO_RN50_MAP50 > 0:
    map_result = 'mAP 높음 ✅ 가설 일치' if YOLO_RN50_MAP50 > YOLO_CSP_MAP50 else 'mAP 낮음 ❌ 가설 불일치'
    fps_result = 'FPS 낮음 ✅ 가설 일치' if YOLO_RN50_FPS   < YOLO_CSP_FPS   else 'FPS 높음 ❌ 가설 불일치'
    print(f'mAP: YOLOv8+RN50({YOLO_RN50_MAP50:.4f}) vs YOLOv8+CSP({YOLO_CSP_MAP50:.4f}) → {map_result}')
    print(f'FPS: YOLOv8+RN50({YOLO_RN50_FPS:.2f}) vs YOLOv8+CSP({YOLO_CSP_FPS:.2f}) → {fps_result}')
else:
    print('학습 완료 후 결과값이 자동으로 입력')

항목                         RetinaNet      YOLOv8+CSP     YOLOv8+RN50
백본                          ResNet50      CSPDarknet        ResNet50
mAP@0.5                       0.5642          0.7740          0.6576
FPS                            12.01           46.25           46.41

=== 가설 2 검증 ===
가설: YOLOv8+ResNet50이 YOLOv8+CSPDarknet보다 mAP↑ FPS↓
mAP: YOLOv8+RN50(0.6576) vs YOLOv8+CSP(0.7740) → mAP 낮음 ❌ 가설 불일치
FPS: YOLOv8+RN50(46.41) vs YOLOv8+CSP(46.25) → FPS 높음 ❌ 가설 불일치


In [12]:
# 비교 차트 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('가설 2 검증: 백본 교체 영향 분석', fontsize=15, fontweight='bold')

models = ['RetinaNet\n(ResNet50)', 'YOLOv8s\n(CSPDarknet)', 'YOLOv8s\n(ResNet50)']
colors = ['#2E75B6', '#E67E22', '#27AE60']

# mAP 비교
ax1 = axes[0]
vals = [RETINA_MAP50, YOLO_CSP_MAP50, YOLO_RN50_MAP50]
bars = ax1.bar(models, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
ax1.set_title('mAP@0.5 비교', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for bar, val in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# FPS 비교
ax2 = axes[1]
vals2 = [RETINA_FPS, YOLO_CSP_FPS, YOLO_RN50_FPS]
bars2 = ax2.bar(models, vals2, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
ax2.set_title('FPS 비교', fontsize=13, fontweight='bold')
ax2.axhline(y=30, color='red', linestyle='--', linewidth=1.5, label='자율주행 최소 기준 (30 FPS)')
ax2.legend(fontsize=9)
ax2.set_ylim(0, max(vals2) * 1.3 if max(vals2) > 0 else 1)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
for bar, val in zip(bars2, vals2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('hypothesis2_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: hypothesis2_chart.png ✅')

/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 51088 (\N{HANGUL SYLLABLE JA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 50984 (\N{HANGUL SYLLABLE YUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 51452 (\N{HANGUL SYLLABLE JU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 54665 (\N{HANGUL SYLLABLE HAENG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_137/1320615530.py:34: UserWarning: Glyph 52572 (\N{HANGUL SYLLABLE COE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


<Figure size 1200x500 with 2 Axes>

차트 저장: hypothesis2_chart.png ✅


## STEP 9. 클래스별 AP 비교 (Pedestrian 중점 분석)

In [13]:
# YOLOv8+ResNet50 클래스별 AP 확인
if os.path.exists(resnet50_ckpt) and 'val_results' in dir():
    print('=== YOLOv8+ResNet50 클래스별 AP ===')
    ap_per_class = val_results.box.ap_class_index
    aps          = val_results.box.ap50
    
    for cls_idx, ap in zip(ap_per_class, aps):
        cls_name = CLASSES[int(cls_idx)]
        print(f'{cls_name:<20}: AP = {ap:.4f}')
    
    print(f'\nmAP@0.5: {resnet50_map50:.4f}')
    
    # RetinaNet 클래스별 AP와 비교
    retina_ap = {
        'Car': 0.8150, 'Van': 0.7628, 'Truck': 0.8654,
        'Pedestrian': 0.0909, 'Person_sitting': 0.2262,
        'Cyclist': 0.3931, 'Tram': 0.7944, 'Misc': 0.5659
    }
    
    print('\n=== Pedestrian AP 비교 (핵심 클래스) ===')
    print(f'RetinaNet  : {retina_ap["Pedestrian"]:.4f}')
    print(f'YOLOv8+CSP : 측정 필요 (YOLOv8 비교분석 노트북 참조)')
    
    # YOLOv8+RN50 Pedestrian AP 찾기
    for cls_idx, ap in zip(ap_per_class, aps):
        if CLASSES[int(cls_idx)] == 'Pedestrian':
            print(f'YOLOv8+RN50: {ap:.4f}')
else:
    print('학습 완료 후 실행')

=== YOLOv8+ResNet50 클래스별 AP ===
Car                 : AP = 0.9119
Van                 : AP = 0.6033
Truck               : AP = 0.8382
Pedestrian          : AP = 0.6581
Person_sitting      : AP = 0.3667
Cyclist             : AP = 0.6455
Tram                : AP = 0.6748
Misc                : AP = 0.5624

mAP@0.5: 0.6576

=== Pedestrian AP 비교 (핵심 클래스) ===
RetinaNet  : 0.0909
YOLOv8+CSP : 측정 필요 (YOLOv8 비교분석 노트북 참조)
YOLOv8+RN50: 0.6581


## STEP 10. 최종 분석 결과 정리

In [14]:
print('=' * 75)
print('최종 3가지 모델 비교 결과')
print('=' * 75)
print(f'{"항목":<22} {"RetinaNet":>16} {"YOLOv8+CSP":>16} {"YOLOv8+RN50":>16}')
print('-' * 75)
print(f'{"백본":<22} {"ResNet50":>16} {"CSPDarknet":>16} {"ResNet50":>16}')
print(f'{"앵커 방식":<22} {"앵커 기반":>16} {"앵커 프리":>16} {"앵커 프리":>16}')
print(f'{"mAP@0.5":<22} {RETINA_MAP50:>16.4f} {YOLO_CSP_MAP50:>16.4f} {YOLO_RN50_MAP50:>16.4f}')
print(f'{"FPS":<22} {RETINA_FPS:>16.2f} {YOLO_CSP_FPS:>16.2f} {YOLO_RN50_FPS:>16.2f}')
print('=' * 75)

print('\n=== 분석 포인트 ===')
print('1. 아키텍처 영향 (①RetinaNet vs ②YOLOv8+CSP):')
print(f'   mAP: {RETINA_MAP50:.4f} → {YOLO_CSP_MAP50:.4f} ({YOLO_CSP_MAP50-RETINA_MAP50:+.4f})')
print(f'   FPS: {RETINA_FPS:.2f} → {YOLO_CSP_FPS:.2f} ({YOLO_CSP_FPS-RETINA_FPS:+.2f})')
print()
print('2. 백본 영향 (②YOLOv8+CSP vs ③YOLOv8+RN50):')
if YOLO_RN50_MAP50 > 0:
    print(f'   mAP: {YOLO_CSP_MAP50:.4f} → {YOLO_RN50_MAP50:.4f} ({YOLO_RN50_MAP50-YOLO_CSP_MAP50:+.4f})')
    print(f'   FPS: {YOLO_CSP_FPS:.2f} → {YOLO_RN50_FPS:.2f} ({YOLO_RN50_FPS-YOLO_CSP_FPS:+.2f})')
else:
    print('   학습 완료 후 결과가 표시')

최종 3가지 모델 비교 결과
항목                            RetinaNet       YOLOv8+CSP      YOLOv8+RN50
---------------------------------------------------------------------------
백본                             ResNet50       CSPDarknet         ResNet50
앵커 방식                             앵커 기반            앵커 프리            앵커 프리
mAP@0.5                          0.5642           0.7740           0.6576
FPS                               12.01            46.25            46.41

=== 분석 포인트 ===
1. 아키텍처 영향 (①RetinaNet vs ②YOLOv8+CSP):
   mAP: 0.5642 → 0.7740 (+0.2098)
   FPS: 12.01 → 46.25 (+34.24)

2. 백본 영향 (②YOLOv8+CSP vs ③YOLOv8+RN50):
   mAP: 0.7740 → 0.6576 (-0.1164)
   FPS: 46.25 → 46.41 (+0.16)
